# GW190521, step by step: Full-sky and Fixed-sky ringdown analysis

The (2,2,0) + (3,3,0) analysis of GW190521 from the paper, written out with the library modules only
(`waveform`, `other_utils`, `swsh`, `strain_cache`), without `analysis.py`:

1. GWOSC strain around the ringdown start, and the arrival times at the fixed IMR sky position.
2. Conditioning (no high-pass, downsampling to 4096 Hz) and 0.2 s analysis windows:
   - **Full-sky:** both detectors start at the arrival time in the last detector (here Hanford);
   - **Fixed-sky:** each detector starts at its own arrival time.
3. Noise covariance from the time-domain autocorrelation of two 64 s off-source stretches.
4. Log-likelihoods, uniform priors, and nested sampling of both models.
5. ln B of (2,2,0) + (3,3,0) over (2,2,0) alone.

The paper used $10^5$ live points (ln B ≈ 2.44 Full-sky, 0.82 Fixed-sky). With $10^4$ a run takes about 20 minutes on a
laptop. Set `NLIVE` (or `SKYRING_NLIVE`) to change it. The strain is read from `strain/` of the Zenodo data if
`SKYRING_DATA` is set; otherwise it is downloaded from GWOSC into `./strain_cache`.

In [ ]:
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=12"   # CPU devices used by the sampler; before importing jax
import sys
sys.path.insert(0, '../src')
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import numpy as onp
import jax
import jax.numpy as jnp
import lal
import numpyro
import numpyro.distributions as dist
from numpyro.contrib.nested_sampling import NestedSampler

import waveform as wf
from swsh import swsh
from other_utils import bandpass_ds, analysis_data, td_acf, cholesky_from_acf, decode_termination
from strain_cache import fetch_or_cache_strain

jax.config.update("jax_enable_x64", True)
numpyro.enable_x64()

In [ ]:
## Settings of the paper's GW190521 runs
t_start = 1242442967.445 + 0.006              # geocentric ringdown start: t_peak + 6 ms (about 4 t_M for M = 325 Msun)
ra, dec = 3.5, 0.73                           # IMR point estimate: sets the arrival times, and the Fixed-sky position
detectors = ("H1", "L1")
seglen, fs, srate, T, t0, trim = 8, 16384, 4096, 0.2, 0.0, 0.25
acf_centers = (t_start - 34, t_start + 34)    # 64 s off-source stretches, ending 2 s before / starting 2 s after t_start
acf_seglen, taper = 64.0, 0.1
modes, modes_220 = ((2, 2, 0), (3, 3, 0)), ((2, 2, 0),)
mass, chi = (150., 500.), (0., 0.99)
amp = {(2, 2, 0): (0., 4.5), (3, 3, 0): (0., 0.6)}   # in units of 1e-20; the likelihood also enforces A330 <= A220
DATA = os.environ.get("SKYRING_DATA")
STRAIN = os.path.join(DATA, "strain") if DATA else "./strain_cache"
NLIVE = int(os.environ.get("SKYRING_NLIVE", 10_000))
SEED = 31567

In [ ]:
## 1. Strain and arrival times
raw = {ifo: fetch_or_cache_strain(ifo, t_start - seglen/2, t_start + seglen/2, fs, STRAIN) for ifo in detectors}
det = {"H1": lal.CachedDetectors[lal.LALDetectorIndexLHODIFF], "L1": lal.CachedDetectors[lal.LALDetectorIndexLLODIFF]}
t_lal = lal.LIGOTimeGPS(t_start)
gmst = float(lal.GreenwichMeanSiderealTime(t_lal))
t_arrival = {ifo: t_start + lal.TimeDelayFromEarthCenter(det[ifo].location, ra, dec, t_lal) for ifo in detectors}
ref_det = max(t_arrival, key=t_arrival.get)            # the last detector to see the signal
t_ref = t_arrival[ref_det]
for ifo in detectors:
    print(f"{ifo}: arrival {t_arrival[ifo]:.6f} ({(t_arrival[ifo] - t_ref) * 1e3:+.3f} ms from {ref_det})")

In [ ]:
## 2. Conditioning and analysis windows (n samples of 0.2 s at 4096 Hz)
n = wf.Ntime(srate, T)
def window(ifo, t_begin):
    """Condition the strain so that a sample falls on t_begin, then take n samples from the one closest to it."""
    times, data = bandpass_ds(raw[ifo], t0=t_begin, ds=fs // srate, trim=trim, f_min=None)   # no high-pass
    return jnp.asarray(analysis_data(data, times, t_begin, n)[1], dtype=jnp.float64)

h_full = {ifo: window(ifo, t_ref) for ifo in detectors}               # Full-sky: common start at t_ref
h_fixed = {ifo: window(ifo, t_arrival[ifo]) for ifo in detectors}     # Fixed-sky: each detector's own arrival
print(f"{n} samples per detector; Full-sky drops the first {(t_ref - t_arrival['L1']) * 1e3:.2f} ms of the L1 ringdown")

In [ ]:
## 3. Noise covariance: average time-domain autocorrelation of the off-source stretches, then its Cholesky factor
acf = {ifo: [] for ifo in detectors}
for c in acf_centers:
    for ifo in detectors:
        stretch = fetch_or_cache_strain(ifo, c - acf_seglen/2, c + acf_seglen/2, fs, STRAIN)
        times, data = bandpass_ds(stretch, t0=c, ds=fs // srate, trim=trim, f_min=None)   # conditioned like the data
        acf[ifo].append(td_acf(data, n, taper=taper))
L = {ifo: cholesky_from_acf(onp.mean(acf[ifo], axis=0)) for ifo in detectors}           # C = L L^T

## detector responses and locations used by the likelihoods
wf.set_detectors(*(jnp.asarray(det[ifo].response, dtype=jnp.float64) for ifo in detectors))
wf.set_detector_locations(*wf.initialize_det_locs())

In [ ]:
## 4. Log-likelihoods (without the parameter-independent normalization) and priors
def swsh_fn(l, m, iota):
    return swsh(2, l, m, iota, 0.0)

def make_loglik(md, sky):
    table = wf.load_qnm_table("../data", md)
    if sky == "full":       # theta = [M, chi, (A, phi) per mode, cos iota, ra, sin dec, psi]
        return lambda th: wf.ln_likelihood_general_full_jit(h_full["H1"], h_full["L1"], th, gmst, L["H1"], L["L1"],
                                                             table, md, swsh_fn, T, srate, t0, ref_det)
    # Fixed-sky: theta = [M, chi, (A, phi) per mode, cos iota, psi], sky fixed at (ra, dec)
    return lambda th: wf.ln_likelihood_general_fixed_sky_prealigned_jit(h_fixed["H1"], h_fixed["L1"], th, gmst,
                                                                         L["H1"], L["L1"], table, md, swsh_fn,
                                                                         ra, dec, T, srate, t0)

def prior_box(md, sky):
    low, high = [mass[0], chi[0]], [mass[1], chi[1]]
    for mode in md:
        low += [amp[mode][0], 0.]; high += [amp[mode][1], 2*onp.pi]           # amplitude, phase
    low.append(-1.); high.append(1.)                                          # cos iota
    if sky == "full":
        low += [0., -1.]; high += [2*onp.pi, 1.]                              # ra, sin dec
    low.append(0.); high.append(onp.pi)                                       # psi
    return jnp.array(low), jnp.array(high)

In [ ]:
## Nested sampling (jaxns through numpyro): stops when ln(Z_dead + Z_live) - ln(Z_dead) < 0.01
def run(md, sky, nlive=NLIVE, seed=SEED):
    loglik, (low, high) = make_loglik(md, sky), prior_box(md, sky)
    def model():
        theta = numpyro.sample("theta", dist.Uniform(low, high))
        numpyro.factor("loglike", loglik(theta))
    rng_run, rng_post = jax.random.split(jax.random.PRNGKey(seed))
    ns = NestedSampler(model, constructor_kwargs=dict(num_live_points=nlive, max_samples=max(500_000, 10 * nlive * len(low)),
                                                      verbose=False),
                       termination_kwargs=dict(dlogZ=0.01))
    ns.run(rng_run)
    r = ns._results
    reasons, converged = decode_termination(int(r.termination_reason))
    theta = onp.asarray(ns.get_samples(rng_post, num_samples=100_000)["theta"])
    print(f"  {sky}-sky {'+'.join(''.join(map(str, m)) for m in md):8s} ln Z = {float(r.log_Z_mean):.2f} +- "
          f"{float(r.log_Z_uncert):.2f} ({'converged' if converged else 'NOT converged: ' + ', '.join(reasons)})")
    return theta, float(r.log_Z_mean)

In [ ]:
## 5. Both models, both sky treatments, and ln B
results = {}
for sky in ("full", "fixed"):
    theta, lnZ = run(modes, sky)
    _, lnZ_220 = run(modes_220, sky)
    ratio = theta[:, 4] / theta[:, 2]                                         # A330 / A220
    results[sky] = dict(theta=theta, lnB=lnZ - lnZ_220)
    print(f"{sky}-sky: ln B = {lnZ - lnZ_220:.2f}; A330/A220 median {onp.median(ratio):.3f} "
          f"[{onp.percentile(ratio, 5):.3f}, {onp.percentile(ratio, 95):.3f}]")